## Checks
  1. Uniqe SEGID in Segment Shapefile
  2. Distance in Shapefile = Calculated Distance (within 0.01 miles)
  3. All *LRS Segments*: `Distance = EMP-BMP` (to nearest tenth of a mile)
  4. All TDM SEGIDs in Segments Shapefile (except exclusions coded into `params/segid-exemptions-notInSegments.csv`)
  5. All Segments Shapefile SEGIDs in TDM (except exclusions coded into `params/segid-exemptions-notInTdm.csv`)
  6. Check coordinate system and (??)

#### For list of segment checks, you can find more info here: 0-How-to-Prep-Segments-for-TDM.ipynb

In [1]:
import _config
import importlib
importlib.reload(_config)
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString

In [2]:
# import master network
gdfWfSegments = gpd.read_file(_config.fnWfSegments, include_fields=['SEGID','BMP','EMP','DISTANCE'])
display(gdfWfSegments)

,SEGID,BMP,EMP,DISTANCE,geometry
0,0006_141.0,141.035,146.868,5.843301,"LINESTRING (405824.110 4423860.330, 406035.300..."
1,0006_146.9,146.868,149.902,3.039023,"LINESTRING (413442.550 4422753.728, 413459.600..."
2,0006_149.9,149.902,150.580,0.677983,"LINESTRING (418330.800 4422866.000, 418629.100..."
3,0006_150.6,150.580,152.555,1.973104,"LINESTRING (419421.880 4422872.296, 419780.300..."
4,0006_152.6,152.555,152.871,0.316275,"LINESTRING (422596.900 4422889.295, 422598.500..."
...,...,...,...,...,...
5475,WFRC_8489,0.000,0.000,0.505514,"LINESTRING (420577.907 4495035.704, 420598.660..."
5476,WFRC_8490,0.000,0.000,0.737896,"LINESTRING (421385.500 4495026.200, 421744.807..."
5477,WFRC_8491,0.000,0.000,0.265495,"LINESTRING (422563.717 4495144.584, 422817.227..."
5478,WFRC_8492,0.000,0.000,0.444725,"LINESTRING (423341.022 4513066.226, 423346.007..."


In [3]:
# import master network
gdfWfTdm = gpd.read_file(_config.fnWfTdm, include_fields=['LINKID','SEGID','LN_2019','LN23_50'])
display(gdfWfTdm)

,LINKID,LN_2019,LN23_50,SEGID,geometry
0,1_29685,7,7,NaN,"LINESTRING (411935.000 4605642.000, 412478.126..."
1,2_29685,7,7,NaN,"LINESTRING (412641.000 4606013.000, 412478.126..."
2,3_29645,7,7,NaN,"LINESTRING (408993.000 4603544.000, 409632.415..."
3,4_29671,7,7,NaN,"LINESTRING (410051.000 4604806.000, 409642.112..."
4,5_29680,7,7,NaN,"LINESTRING (413394.000 4603600.000, 413482.000..."
...,...,...,...,...,...
57112,95064_95025,1,1,2865_019.4,"LINESTRING (473181.550 4444171.200, 473172.240..."
57113,95064_95026,1,1,2865_019.4,"LINESTRING (473181.550 4444171.200, 473177.690..."
57114,95064_95056,1,1,NaN,"LINESTRING (473181.550 4444171.200, 473196.790..."
57115,95065_3621,7,7,NaN,"LINESTRING (480730.710 4444037.000, 480767.876..."


# Check #1: Uniqe SEGID in Segment Shapefile

In [4]:
# Assuming gdfWfSegments is your DataFrame and 'SEGID' is the column of interest
# This will return a Series with True for each row that is a duplicate in the 'SEGID' column
duplicates = gdfWfSegments.duplicated(subset=['SEGID'], keep=False)

# To see the duplicated rows
duplicated_rows = gdfWfSegments[duplicates].sort_values(by='SEGID')

# If you want to count the number of duplicates
number_of_duplicates = duplicates.sum()

if number_of_duplicates>0:
    print(f'Number of duplicate SEGID entries: {number_of_duplicates}')
    display (duplicated_rows[['SEGID','BMP','EMP','DISTANCE']])
else:
    print("CHECK #1 COMPLETE. NO DUPLICATES... YOU PASSED!!")


CHECK #1 COMPLETE. NO DUPLICATES... YOU PASSED!!


# Check #2: Distance in Shapefile = Calculated Distance


In [5]:
# Project to a UTM zone appropriate for your data, e.g., EPSG:32633 for UTM zone 33N
# You'll need to find the UTM zone appropriate for your GeoDataFrame's location
gdf_projected = gdfWfSegments.to_crs(epsg=32612)

gdf_projected['actual_distance'] = gdf_projected.length * 0.000621371

# Assuming gdfWfSegments is your GeoDataFrame and it already includes 'actual_distance' column
# Calculate the absolute difference between DISTANCE and actual_distance
gdf_projected['distance_difference'] = abs(gdf_projected['DISTANCE'] - gdf_projected['actual_distance'])

# Filter records where the difference is greater than 0.01 miles
records_off_by_more_than_point_O_one = gdf_projected[gdf_projected['distance_difference'] > 0.01]

# Display the number of such records
number_of_records = len(records_off_by_more_than_point_O_one)

if number_of_records > 0: 
    print(f"Number of records where DISTANCE is off by more than 0.01 miles: {number_of_records}")
    display(records_off_by_more_than_point_O_one)
else:
    print("CHECK #2 COMPLETE. DISTANCES ARE WITHIN 0.01 MILES... YOU PASSED!!")

Number of records where DISTANCE is off by more than 0.01 miles: 2


,SEGID,BMP,EMP,DISTANCE,geometry,actual_distance,distance_difference
5211,WFRC_8217,0.0,0.0,0.583552,"LINESTRING (416980.986 4482553.805, 416719.334...",0.433486,0.150066
5479,WFRC_8493,0.0,0.0,0.583552,"LINESTRING (417222.490 4482555.470, 416980.986...",0.150067,0.433485


# Check #3. All *LRS Segments*: `Distance = EMP-BMP` (to nearest tenth of a mile)


In [6]:
# Project to a UTM zone appropriate for your data, e.g., EPSG:32633 for UTM zone 33N
# You'll need to find the UTM zone appropriate for your GeoDataFrame's location
gdf_projected = gdfWfSegments.to_crs(epsg=32612)

gdf_projected['actual_distance'] = gdf_projected.length * 0.000621371

# Assuming gdfWfSegments is your GeoDataFrame and it already includes 'actual_distance' column
# Calculate the absolute difference between DISTANCE and actual_distance
gdf_projected.loc[gdf_projected['EMP'] > 0, 'distance_difference'] = abs(gdf_projected['EMP'] - gdf_projected['BMP'] - gdf_projected['actual_distance'])

# Filter records where the difference is greater than 0.01 miles
records_off_by_more_than_point_O_one = gdf_projected[gdf_projected['distance_difference'] > 0.01]

# Display the number of such records
number_of_records = len(records_off_by_more_than_point_O_one)

if number_of_records > 0: 
    print(f"Number of records where DISTANCE is off by more than 0.01 miles from BMP-EMP: {number_of_records}")
    display(records_off_by_more_than_point_O_one)
else:
    print("CHECK #3 COMPLETE. DISTANCES ARE WITHIN 0.01 MILES... YOU PASSED!!")

Number of records where DISTANCE is off by more than 0.01 miles from BMP-EMP: 682


,SEGID,BMP,EMP,DISTANCE,geometry,actual_distance,distance_difference
0,0006_141.0,141.035,146.868,5.843301,"LINESTRING (405824.110 4423860.330, 406035.300...",5.843311,0.010311
6,0006_155.8,155.793,155.935,0.201632,"LINESTRING (427206.400 4424885.380, 427217.600...",0.201632,0.059632
7,0006_155.9,155.935,157.295,1.316023,"LINESTRING (427434.740 4425115.940, 427492.600...",1.316025,0.043975
16,0006_173.4,173.424,173.716,0.446194,"LINESTRING (445161.200 4441631.100, 444987.240...",0.446195,0.154195
20,0006_174.9,174.852,175.574,0.699839,"LINESTRING (447081.015 4439469.207, 446942.800...",0.699840,0.022160
...,...,...,...,...,...,...,...
4088,3462_002.8,2.772,5.405,2.594257,"LINESTRING (421389.000 4574522.150, 421430.900...",2.594261,0.038739
4090,3466_000.0,0.000,0.104,0.115698,"LINESTRING (416815.520 4574743.140, 416836.040...",0.115699,0.011699
4093,3470_000.0,0.000,1.026,1.039352,"LINESTRING (416028.260 4573305.310, 416031.640...",1.039354,0.013354
4882,MAG_6855,0.385,1.081,0.458597,"LINESTRING (448232.937 4442435.494, 448398.501...",0.458598,0.237402


### IGNORE CHECK #3 FOR NOW, SINCE NOT CRITICAL.... PICK UP LATER! STILL IMPORTANT TO MAKE SURE CONSISTENT

# Prep for 4 and 5

In [7]:
# get SegId Concordance

# Remove duplicate SEGID records from gdfWfTdm
gdfWfTdm_grouped = gdfWfTdm.groupby(['SEGID'], as_index=False).agg(numLinks=('LINKID','count'),firstLinkId=('LINKID','first'),firstLanes2019=('LN_2019','first'),firstLanes2050=('LN23_50','first'))

# Merge the two dataframes with an outer join to find all unique 'SEGID' values
merged_df = pd.merge(gdfWfTdm_grouped, gdfWfSegments['SEGID'], on='SEGID', how='outer', indicator=True)

dfUniqueToTdm = merged_df[merged_df['_merge']=='left_only']
dfUniqueToSegments = merged_df[merged_df['_merge']=='right_only']

dfExemptionsNotInSegments = pd.read_csv(_config.fnExemptionsNotInSegments)
dfExemptionsNotInTdm = pd.read_csv(_config.fnExemptionsNotInTdm)


# Check #4. All TDM SEGIDs in Segments Shapefile (except exclusions coded into `params/segid-exemptions-notInSegments.csv`)


In [8]:
# Ensure SEGID is the column name in both dataframes
dfUniqueToTdmFiltered = dfUniqueToTdm[~dfUniqueToTdm['SEGID'].isin(dfExemptionsNotInSegments['SEGID'])]

# Display the number of such records
dfUniqueToTdmFiltered_len = len(dfUniqueToTdmFiltered)

if dfUniqueToTdmFiltered_len > 0: 
    print(f"Number of TDM SEGIDs that are not in Segment Shapefile: {dfUniqueToTdmFiltered_len}")
    display(dfUniqueToTdmFiltered)
else:
    print("CHECK #4 COMPLETE. NO MISSING TDM SEGIDS... YOU PASSED!!")


CHECK #4 COMPLETE. NO MISSING TDM SEGIDS... YOU PASSED!!


# Check #5. All Segments Shapefile SEGIDs in TDM (except exclusions coded into `params/segid-exemptions-notInTdm.csv`)


In [9]:
# Ensure SEGID is the column name in both dataframes
dfUniqueToSegmentsFiltered = dfUniqueToSegments[~dfUniqueToSegments['SEGID'].isin(dfExemptionsNotInTdm['SEGID'])]

dfUniqueToSegmentsFiltered = dfUniqueToSegmentsFiltered[['SEGID']].drop_duplicates()

# Display the number of such records
dfUniqueToSegmentsFiltered_len = len(dfUniqueToSegmentsFiltered)

if dfUniqueToSegmentsFiltered_len > 0: 
    print(f"Number of SEGIDs that are not in TDM: {dfUniqueToSegmentsFiltered_len}")
    display(dfUniqueToSegmentsFiltered)
else:
    print("CHECK #5 COMPLETE. NO MISSING SEGMENT SEGIDS... YOU PASSED!!")


Number of SEGIDs that are not in TDM: 1


,SEGID
5479,WFRC_8493


# CHECK #6: CRS

In [10]:
# DO MORE LATER